# 1. LLM 101

This notebook shows a minimal LangChain call with Gemini.

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv() # Loading API credentials from .env file

: 

In [ ]:
# Setting the agent
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
)

: 

In [4]:
# Prompt that illustrate how we discovered that we could use tool
fake_system_prompt = """You are a helpful assistant and you call tools when required.

Here is the list of tools you have access to and how to call them:

# Tool 1: Capital finder
Given the <country>, it returns the capital city.

To call the tool, return:

```json
{
  "tool_name": "capital_finder",
  "args": {
    "country": "<country>"
  }
}
```

# Tool 2: Weather finder
Given the <city>, it returns the current weather.

To call the tool, return:

```json
{
  "tool_name": "weather_finder",
  "args": {
    "city": "<city>"
  }
}
```"""


In [5]:
fake_user_prompt = "What is the capital of Belgium? And what's the weather in Paris?"
final_prompt = f"""{fake_system_prompt}
User Query: {fake_user_prompt}"""
response = llm.invoke(final_prompt)
print(response)
print(response.content[0].get("text"))

content=[{'type': 'text', 'text': '```json\n{\n  "tool_name": "capital_finder",\n  "args": {\n    "country": "Belgium"\n  }\n}\n```\n\n```json\n{\n  "tool_name": "weather_finder",\n  "args": {\n    "city": "Paris"\n  }\n}\n```', 'extras': {'signature': 'EjQKMgEMOdbHuYNrLB1jMB45ZrROmSlh5ue+ZZ1fWpXNNZ5JE9AHE15nYmxAGgbkWNyrtNke'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019ef8bd-bef2-7382-bd2d-15a980c0375b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 182, 'output_tokens': 75, 'total_tokens': 257, 'input_token_details': {'cache_read': 0}}
```json
{
  "tool_name": "capital_finder",
  "args": {
    "country": "Belgium"
  }
}
```

```json
{
  "tool_name": "weather_finder",
  "args": {
    "city": "Paris"
  }
}
```


## The same idea with `with_structured_output`

Instead of manually describing a JSON format in the prompt, we can define the expected tool arguments with Pydantic and let LangChain enforce the structure.

In [6]:
from pydantic import BaseModel, Field


class CapitalFinderArgs(BaseModel):
    country: str = Field(description="The country for which we want the capital city.")


class WeatherFinderArgs(BaseModel):
    city: str = Field(description="The city for which we want the current weather.")


class ToolSelection(BaseModel):
    capital_finder: CapitalFinderArgs | None = Field(
        default=None,
        description="Arguments for the capital finder tool when needed.",
    )
    weather_finder: WeatherFinderArgs | None = Field(
        default=None,
        description="Arguments for the weather finder tool when needed.",
    )

In [7]:
structured_llm = llm.with_structured_output(ToolSelection)

structured_response = structured_llm.invoke(
    f"Identify which tool arguments are needed for this user query: {fake_user_prompt}"
)

structured_response

ToolSelection(capital_finder=CapitalFinderArgs(country='Belgium'), weather_finder=WeatherFinderArgs(city='Paris'))

In [8]:
tool_calls = []

if structured_response.capital_finder:
    tool_calls.append(
        {
            "tool_name": "capital_finder",
            "args": structured_response.capital_finder.model_dump(),
        }
    )

if structured_response.weather_finder:
    tool_calls.append(
        {
            "tool_name": "weather_finder",
            "args": structured_response.weather_finder.model_dump(),
        }
    )

tool_calls

[{'tool_name': 'capital_finder', 'args': {'country': 'Belgium'}},
 {'tool_name': 'weather_finder', 'args': {'city': 'Paris'}}]

## How to stream answer with langchain

In [9]:
prompt = "Tell me in details what LangChain is in 5 lines."

print("Streaming response:\n")

for chunk in llm.stream(prompt):
    print(chunk.text, end="", flush=True)

Streaming response:

LangChain is an open-source framework designed to simplify the creation of applications powered by Large Language Models (LLMs).
It provides a modular architecture that allows developers to chain together various components, such as prompts, models, and external data sources.
By enabling "chains," it allows LLMs to perform complex workflows, like retrieving private documents or executing multi-step reasoning tasks.
The framework acts as a bridge, connecting static AI models to real-time data and external tools like APIs, databases, and search engines.
Ultimately, it streamlines the development of sophisticated, context-aware AI agents that can interact dynamically with the world.